<a href="https://colab.research.google.com/github/Sagnik-Chowdhury/Federated-Learning-1/blob/Sourit/Fed_Gradient_Clipping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Clipping & Differential Privacy (DP-SGD)
### Analyzing Privacy Noise Across Data Modalities

**Objective:**
This notebook simulates an industry-standard Differential Privacy pipeline (DP-SGD) in a Federated Learning environment. We are testing how restricting client updates (Clipping) and injecting statistical noise (Laplace vs. Gaussian) impacts different data structures.

**The Hypothesis:**
Dense, highly integrated data (like MNIST pixels) relies on fragile spatial correlations that are easily destroyed by heavy-tailed privacy noise. Scattered, tabular data (like Breast Cancer features) is hypothesized to be much more robust to Differential Privacy mechanisms, specifically Laplace noise.

## Libraries

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
import copy
import pandas as pd
import math

## Architecture

In [16]:
#  For Integrated Data (MNIST Images)
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

In [17]:
class TabularNet(nn.Module):
    def __init__(self):
        super(TabularNet, self).__init__()
        # Reduced capacity: 30 -> 16 -> 8 -> 2
        self.fc1 = nn.Linear(30, 16)
        # Dropout randomly turns off 20% of neurons to prevent memorization
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x) # Apply dropout after the first layer
        x = self.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
# For Scattered Data (Tabular Features)
# The Breast Cancer dataset has 30 input features and 2 output classes
class TabularNet(nn.Module):
    def __init__(self):
        super(TabularNet, self).__init__()
        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

## Data Preparation

In [18]:
NUM_CLIENTS = 20

In [19]:
print("Preparing MNIST Dataset (Integrated Data)")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

mnist_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_split = random_split(mnist_full, [len(mnist_full) // NUM_CLIENTS] * NUM_CLIENTS)
mnist_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in mnist_split]

mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transform)
mnist_test_loader = DataLoader(mnist_test, batch_size=1000, shuffle=False)

print("\nData preparation complete")

Preparing MNIST Dataset (Integrated Data)

Data preparation complete


In [20]:
from sklearn.model_selection import train_test_split

print("Preparing Breast Cancer Dataset (Strict Train/Test Split)...")

# 1. Load and scale the data
data = load_breast_cancer()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data.data)
y = data.target

# 2. THE FIX: Strictly separate 20% of the data for testing BEFORE tensor conversion
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 3. Convert to PyTorch Tensors
tabular_train = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
tabular_test = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

# 4. Split ONLY the training data among clients
tab_split_size = len(tabular_train) // NUM_CLIENTS
tab_splits = [tab_split_size] * NUM_CLIENTS
tab_splits[-1] += len(tabular_train) % NUM_CLIENTS
tabular_loaders = [DataLoader(ds, batch_size=8, shuffle=True) for ds in random_split(tabular_train, tab_splits)]

# 5. The Test Loader now holds data the clients have NEVER seen
tabular_test_loader = DataLoader(tabular_test, batch_size=len(tabular_test), shuffle=False)

print("\nData preparation complete!")

Preparing Breast Cancer Dataset (Strict Train/Test Split)...

Data preparation complete!


In [ ]:
print("Preparing Breast Cancer Dataset (Scattered/Tabular Data)")

# Load and scale tabular data
data = load_breast_cancer()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data.data)
y = data.target

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
tabular_full = TensorDataset(X_tensor, y_tensor)

# Split tabular data among clients
# Tabular datasets are much smaller, so batch size is reduced
tab_split_size = len(tabular_full) // NUM_CLIENTS
# Handle remainders if dataset doesn't divide perfectly
tab_splits = [tab_split_size] * NUM_CLIENTS
tab_splits[-1] += len(tabular_full) % NUM_CLIENTS
tabular_loaders = [DataLoader(ds, batch_size=8, shuffle=True) for ds in random_split(tabular_full, tab_splits)]
tabular_test_loader = DataLoader(tabular_full, batch_size=len(tabular_full), shuffle=False)

print("\nData preparation complete")

Preparing Breast Cancer Dataset (Scattered/Tabular Data)

Data preparation complete


## The Defense Mechanism: Gradient Clipping (L2 Norm)

Unlike Trimmed Mean which discards entire clients based on quantiles, **Gradient Clipping** acts as a strict mathematical speed limit for every participating client.

**How it works:**
1. The server calculates the mathematical difference between the global model and the client's proposed update.
2. It calculates the magnitude (**L2 Norm**) of that update.
3. If the magnitude exceeds our strict `clip_threshold` (set to 2.0), the update is mathematically scaled down so it cannot overpower the global model.
4. Finally, statistical noise (Normal or Laplace) is added to the clipped average to guarantee Differential Privacy.

In [21]:
def absolute_weight_clipping_aggregation(client_weights_list, clip_threshold=1.0):
    """
    Calculates the L2 norm of the ENTIRE client weight vector (absolute position)
    and scales the whole model down if it exceeds the threshold.
    """
    clipped_weights_list = []

    # 1. Calculate and clip each client's absolute weights
    for client_weights in client_weights_list:
        squared_sum = 0.0

        # Calculate the L2 Norm (magnitude) of the ENTIRE model
        for key in client_weights.keys():
            squared_sum += torch.sum(client_weights[key] ** 2).item()

        l2_norm = math.sqrt(squared_sum)

        # Calculate the scaling factor (if norm > threshold, scale > 1)
        scale = max(1.0, l2_norm / clip_threshold)

        # Apply the scale down to the absolute weights
        clipped_client_weights = {}
        for key in client_weights.keys():
            clipped_client_weights[key] = client_weights[key] / scale

        clipped_weights_list.append(clipped_client_weights)

    # 2. Average the safely clipped absolute weights
    # Grab the first client's dictionary as a template
    avg_weights = copy.deepcopy(clipped_weights_list[0])

    for key in avg_weights.keys():
        stacked_weights = torch.stack([client[key] for client in clipped_weights_list])
        avg_weights[key] = torch.mean(stacked_weights, dim=0)

    # Note: We do not add this back to global_weights.
    # Because we averaged absolute positions, this IS the new global model.
    return avg_weights

In [22]:
def add_dp_noise(weights, noise_type='none', scale=0.01):
    """
    Injects Differential Privacy noise into the aggregated weights.
    Compares Laplace (good for sparse/scattered) vs Normal (Gaussian).
    """
    if noise_type == 'none':
        return weights

    noisy_weights = copy.deepcopy(weights)

    for key in noisy_weights.keys():
        tensor = noisy_weights[key]

        if noise_type == 'normal':
            noise = torch.randn_like(tensor) * scale
        elif noise_type == 'laplace':
            m = torch.distributions.laplace.Laplace(torch.tensor([0.0]), torch.tensor([scale]))
            noise = m.sample(tensor.shape).squeeze(-1).to(tensor.device)

        noisy_weights[key] = tensor + noise

    return noisy_weights

## Automated Grid Search Execution

The following loop executes a completely automated grid search across two data modalities (Image vs. Tabular) and three privacy states (Baseline, Gaussian Noise, and Laplace Noise).

* **Federated Rounds:** 5
* **Clients:** 20
* **DP Noise Scale:** 0.05
* **Clipping Threshold:** 2.0

In [23]:
datasets_to_test = ['mnist', 'tabular']
noise_types_to_test = ['none', 'normal', 'laplace']
NOISE_SCALE = 0.05
federated_rounds = 5
epochs_per_round = 1

experiment_results = {'mnist': {}, 'tabular': {}}

print("Starting Automated Grid Search for Gradient Clipping")

for dataset in datasets_to_test:
    for noise in noise_types_to_test:
        print(f"\nTesting {dataset.upper()} with {noise.upper()} noise")

        # 1. SETUP & RESET THE MODEL
        if dataset == 'mnist':
            global_model = MNISTNet()
            loaders = mnist_loaders
            test_loader = mnist_test_loader
            lr = 0.001
        else:
            global_model = TabularNet()
            loaders = tabular_loaders
            test_loader = tabular_test_loader
            lr = 0.01

        # 2. THE EXECUTION LOOP (Training)
        for round_num in range(federated_rounds):
            client_weights = []

            for client_idx in range(NUM_CLIENTS):
                local_model = MNISTNet() if dataset == 'mnist' else TabularNet()
                local_model.load_state_dict(global_model.state_dict())

                optimizer = optim.Adam(local_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()

                local_model.train()
                for epoch in range(epochs_per_round):
                    for inputs, labels in loaders[client_idx]:
                        optimizer.zero_grad()
                        outputs = local_model(inputs)
                        loss = criterion(outputs, labels)
                        loss.backward()
                        optimizer.step()

                client_weights.append(local_model.state_dict())

            # --- TASK 3 MODIFICATION: GRADIENT CLIPPING ---
            aggregated_weights = gradient_clipping_aggregation(
                client_weights,
                global_model.state_dict(),
                clip_threshold=2.0
            )

            # Add the DP Noise
            secured_weights = add_dp_noise(aggregated_weights, noise_type=noise, scale=NOISE_SCALE)
            global_model.load_state_dict(secured_weights)

        # 3. THE EVALUATION (Testing)
        global_model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = global_model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        final_accuracy = 100 * correct / total
        print(f" Accuracy: {final_accuracy:.2f}% ")

        experiment_results[dataset][noise] = round(final_accuracy,3)


Starting Automated Grid Search for Gradient Clipping

Testing MNIST with NONE noise
 Accuracy: 90.70% 

Testing MNIST with NORMAL noise
 Accuracy: 58.44% 

Testing MNIST with LAPLACE noise
 Accuracy: 25.45% 

Testing TABULAR with NONE noise
 Accuracy: 95.61% 

Testing TABULAR with NORMAL noise
 Accuracy: 97.37% 

Testing TABULAR with LAPLACE noise
 Accuracy: 93.86% 


In [24]:
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

results_df = pd.DataFrame(experiment_results).T
results_df.columns = ['Baseline (No DP)', 'Normal (Gaussian)', 'Laplace']
results_df.index = ['MNIST (Dense)', 'Breast Cancer (Scattered)']

print(results_df.to_string())
print("="*50)


FINAL RESULTS
                           Baseline (No DP)  Normal (Gaussian)  Laplace
MNIST (Dense)                        90.700             58.440    25.45
Breast Cancer (Scattered)            95.614             97.368    93.86


## Final Observations & Conclusion

The experiment yielded dramatic, conclusive results regarding data modality and Differential Privacy:

1. **Integrated Data is Fragile:** The MNIST model suffered catastrophic forgetting when exposed to heavy-tailed noise, dropping from a baseline of **90%** down to **58%** (Gaussian) and plummeting to **25%** (Laplace).

2. **Scattered Data is Robust:** The Breast Cancer tabular model showed incredible resilience. It maintained a **97%** accuracy under Gaussian noise and a highly stable **94%** under the heavier Laplace noise.

**Conclusion:** The structure of the data dictates the defense. Federated models handling scattered tabular features can easily survive aggressive, heavy-tailed privacy mechanisms like Laplace distributions. However, applying that same mathematical noise to dense image data destroys the learned spatial patterns.